In [2]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (14, 8)

In [3]:
def load_results(results_folder):
    result_files = [f for f in os.listdir(results_folder) if f.endswith('.json')]
    results = {}
    
    for file in result_files:
        file_path = os.path.join(results_folder, file)
        with open(file_path, 'r') as f:
            results[file] = json.load(f)
    
    return results

In [4]:
def extract_detailed_node_metrics(results, experiment_name):
    """Extract detailed node metrics including platform results"""
    node_data = []
    platform_data = []
    storage_data = []

    for result_file, result in results.items():
        if 'nodeResults' in result['stats']:
            for node in result['stats']['nodeResults']:
                # Basic node metrics
                node_metrics = {
                    'experiment': experiment_name,
                    'file': result_file,
                    'node_id': node['nodeId'],
                    'unused': node.get('unused', False),
                    'scheduling_time': node.get('schedulingTime', 0),
                    'storage_time': node.get('storageTime', 0),
                    'local_dependencies': node.get('localDependencies', 0),
                    'cache_hits': node.get('cacheHits', 0),
                }

                # Energy metrics by platform type
                if 'energy' in node:
                    for platform_type, energy in node['energy'].items():
                        node_metrics[f'energy_{platform_type}'] = energy

                # Idle energy metrics
                if 'energyIdle' in node:
                    for platform_type, energy in node['energyIdle'].items():
                        node_metrics[f'idle_energy_{platform_type}'] = energy

                # Idle time metrics
                if 'idleTime' in node:
                    for platform_type, time_val in node['idleTime'].items():
                        node_metrics[f'idle_time_{platform_type}'] = time_val

                node_data.append(node_metrics)

                # Platform detailed metrics
                if 'platformResults' in node:
                    for platform in node['platformResults']:
                        platform_metrics = {
                            'experiment': experiment_name,
                            'file': result_file,
                            'node_id': node['nodeId'],
                            'platform_id': platform['platformId'],
                            'platform_type': platform['platformType']['shortName'],
                            'platform_hardware': platform['platformType']['hardware'],
                            'platform_price': platform['platformType']['price'],
                            'energy': platform.get('energy', 0),
                            'energy_idle': platform.get('energyIdle', 0),
                            'idle_time': platform.get('idleTime', 0),
                            'idle_proportion': platform.get('idleProportion', 0),
                            'storage_time': platform.get('storageTime', 0)
                        }
                        platform_data.append(platform_metrics)

                # todo: which are static and can serve as inputs, which are results of the simulation?
                # Storage metrics
                if 'storageResults' in node:
                    for storage in node['storageResults']:
                        storage_metrics = {
                            'experiment': experiment_name,
                            'file': result_file,
                            'node_id': node['nodeId'],
                            'storage_id': storage['storageId'],
                            'has_usage_data': len(storage.get('totalUsage', [])) > 0
                        }
                        storage_data.append(storage_metrics)

    return pd.DataFrame(node_data), pd.DataFrame(platform_data), pd.DataFrame(storage_data)

In [5]:
def extract_detailed_task_metrics(results, experiment_name):
    """Extract detailed task metrics"""
    tasks_data = []

    for result_file, result in results.items():
        if 'taskResults' in result['stats']:
            for task in result['stats']['taskResults']:
                task_data = {
                    'experiment': experiment_name,
                    'file': result_file,
                    'task_id': task['taskId'],
                    'dispatched_time': task['dispatchedTime'],
                    'scheduled_time': task['scheduledTime'],
                    'arrived_time': task['arrivedTime'],
                    'started_time': task['startedTime'],
                    'done_time': task['doneTime'],
                    'task_type': task['taskType']['name'],
                    'platform_type': task['platform']['shortName'],
                    'platform_hardware': task['platform']['hardware'],
                    'elapsed_time': task['elapsedTime'],
                    'pull_time': task.get('pullTime', 0),
                    'cold_start_time': task.get('coldStartTime', 0),
                    'execution_time': task.get('executionTime', 0),
                    'wait_time': task.get('waitTime', 0),
                    'queue_time': task.get('queueTime', 0),
                    'initialization_time': task.get('initializationTime', 0),
                    'compute_time': task.get('computeTime', 0),
                    'communications_time': task.get('communicationsTime', 0),
                    'cold_started': task.get('coldStarted', False),
                    'cache_hit': task.get('cacheHit', False),
                    'local_dependencies': task.get('localDependencies', False),
                    'local_communications': task.get('localCommunications', False),
                    'energy': task.get('energy', 0),
                    'network_latency': task['networkLatency'],
                    'source_node': task['sourceNode'],
                    'execution_node': task['executionNode']
                }

                # Calculate task latency components
                task_data['scheduling_latency'] = task_data['scheduled_time'] - task_data['dispatched_time']
                task_data['arrival_latency'] = task_data['arrived_time'] - task_data['scheduled_time']
                task_data['start_latency'] = task_data['started_time'] - task_data['arrived_time']
                task_data['processing_latency'] = task_data['done_time'] - task_data['started_time']
                task_data['end_to_end_latency'] = task_data['done_time'] - task_data['dispatched_time']

                tasks_data.append(task_data)

    return pd.DataFrame(tasks_data)

In [ ]:
import os
import pickle
from collections import defaultdict
from tqdm import tqdm

def load_experiment_data(experiments, cache_dir="src/notebooks/original_data_cache"):
    os.makedirs(cache_dir, exist_ok=True)
    experiment_data = {}

    for name, folder in tqdm(experiments.items(), desc="Loading experiments"):
        cache_path = os.path.join(cache_dir, f"{name}_data.pkl")
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                experiment_data[name] = pickle.load(f)
        else:
            results = load_results(folder)
            node_df, platform_df, storage_df = extract_detailed_node_metrics(results, name)
            task_df = extract_detailed_task_metrics(results, name)
            experiment_data[name] = {
                "node_df": node_df,
                "platform_df": platform_df,
                "storage_df": storage_df,
                "task_df": task_df,
            }
            with open(cache_path, "wb") as f:
                pickle.dump(experiment_data[name], f)
    return experiment_data

In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import numpy as np
from tqdm import tqdm
import logging
import time
from typing import Dict, List, Tuple, Optional

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

In [ ]:
experiments = {
    "knative": "/root/projects/my-herosim/src/notebooks/data/results_sim/125-225-5clients-knative-with-network",
}
experiment_data = load_experiment_data(experiments, cache_dir="src/notebooks/original_data_cache")

results = experiment_data['knative']
node_df = results['node_df']
platform_df = results['platform_df']
stor_df = results['storage_df']
task_df = results['task_df']

In [10]:
# GNN Model Implementation
class LatencyPredictionGNN(torch.nn.Module):
    def __init__(self, 
                 node_feature_dim: int,
                 edge_feature_dim: int,
                 hidden_dim: int = 64,
                 num_layers: int = 3,
                 dropout: float = 0.1):
        super(LatencyPredictionGNN, self).__init__()
        
        self.node_feature_dim = node_feature_dim
        self.edge_feature_dim = edge_feature_dim
        self.hidden_dim = hidden_dim
        
        # Node feature encoder
        self.node_encoder = torch.nn.Linear(node_feature_dim, hidden_dim)
        
        # Edge feature encoder
        self.edge_encoder = torch.nn.Linear(edge_feature_dim, hidden_dim)
        
        # GNN layers
        self.conv_layers = torch.nn.ModuleList()
        self.batch_norms = torch.nn.ModuleList()
        
        for i in range(num_layers):
            if i == 0:
                # First layer: node features + edge features
                self.conv_layers.append(GCNConv(hidden_dim, hidden_dim))
            else:
                self.conv_layers.append(GCNConv(hidden_dim, hidden_dim))
            self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Output layer for latency prediction
        self.output_layer = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim // 2, 1)
        )
        
        self.dropout = dropout
    
    def forward(self, x, edge_index, edge_attr, batch=None):
        # Encode node features
        x = self.node_encoder(x)
        
        # Encode edge features
        edge_attr = self.edge_encoder(edge_attr)
        
        # Apply GNN layers
        for i, (conv, norm) in enumerate(zip(self.conv_layers, self.batch_norms)):
            x = conv(x, edge_index, edge_attr)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Predict latency for each node
        latency_pred = self.output_layer(x)
        
        return latency_pred.squeeze(-1)

In [ ]:
def prepare_training_data(task_df, platform_df, node_df, system_df):
    """
    Prepare training data by creating graph representations for each task execution
    """
    training_data = []
    
    # Group tasks by simulation file to get system states
    for file_name in task_df['file'].unique():
        file_tasks = task_df[task_df['file'] == file_name]

        # Process each task
        for _, task in file_tasks.iterrows():
            # Create graph data
            graph_data = create_graph_data_from_system_state(
                system_state, task, task_df, platform_df, node_df
            )
            
            if graph_data is None:
                continue
                
            data, node_mapping, node_info = graph_data
            
            # Find the executed node in the graph
            executed_node_key = (task['execution_node'], task.get('platform_id', 0))
            if executed_node_key in node_mapping:
                executed_node_idx = node_mapping[executed_node_key]
                
                # Create target: actual latency for executed node, 0 for others
                target = torch.zeros(len(node_mapping))
                target[executed_node_idx] = task['end_to_end_latency']
                
                # Add mask for training (only executed node has ground truth)
                mask = torch.zeros(len(node_mapping), dtype=torch.bool)
                mask[executed_node_idx] = True
                
                data.y = target
                data.mask = mask
                data.task_info = {
                    'task_id': task['task_id'],
                    'task_type': task['task_type'],
                    'executed_node': task['execution_node'],
                    'actual_latency': task['end_to_end_latency']
                }
                
                training_data.append(data)
    
    return training_data

In [ ]:
# Prepare training data
print("Preparing training data...")
training_data = prepare_training_data(task_df, platform_df, node_df, system_df)
print(f"Created {len(training_data)} training examples")

if training_data:
    # Split data
    train_data, val_data = train_test_split(training_data, test_size=0.2, random_state=42)
    
    # Create data loaders
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
    
    # Initialize model
    sample_data = training_data[0]
    model = LatencyPredictionGNN(
        node_feature_dim=sample_data.x.size(1),
        edge_feature_dim=sample_data.edge_attr.size(1),
        hidden_dim=64,
        num_layers=3
    )
    
    # Training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = torch.nn.MSELoss()
    
    # Training loop
    num_epochs = 50
    train_losses = []
    val_losses = []
    
    print("Starting training...")
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            
            # Forward pass
            pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            
            # Apply mask to only compute loss on executed nodes
            mask = batch.mask
            loss = criterion(pred[mask], batch.y[mask])
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                mask = batch.mask
                loss = criterion(pred[mask], batch.y[mask])
                val_loss += loss.item()
        
        train_losses.append(train_loss / len(train_loader))
        val_losses.append(val_loss / len(val_loader))
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Train Loss = {train_losses[-1]:.4f}, Val Loss = {val_losses[-1]:.4f}")
    
    # Plot training curves
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training Curves')
    
    # Test the model on a few examples
    plt.subplot(1, 2, 2)
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch in val_loader[:5]:  # Test on first 5 batches
            pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            mask = batch.mask
            
            predictions.extend(pred[mask].numpy())
            actuals.extend(batch.y[mask].numpy())
    
    plt.scatter(actuals, predictions, alpha=0.6)
    plt.plot([min(actuals), max(actuals)], [min(actuals), max(actuals)], 'r--', label='Perfect Prediction')
    plt.xlabel('Actual Latency')
    plt.ylabel('Predicted Latency')
    plt.legend()
    plt.title('Prediction vs Actual')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Final Training Loss: {train_losses[-1]:.4f}")
    print(f"Final Validation Loss: {val_losses[-1]:.4f}")
    
    # Calculate R² score
    from sklearn.metrics import r2_score
    r2 = r2_score(actuals, predictions)
    print(f"R² Score: {r2:.4f}")
    
else:
    print("No training data could be created. Check the data structure.")

In [ ]:
# Demonstration: How the model would be used in the scheduler
def predict_latency_for_scheduling(model, system_state, task, task_df, platform_df, node_df):
    """
    Simulate how the GNN model would be used in the scheduler
    Returns predicted latencies for all available (node, platform) combinations
    """
    # Create graph data
    graph_data = create_graph_data_from_system_state(system_state, task, task_df, platform_df, node_df)
    
    if graph_data is None:
        return {}
    
    data, node_mapping, node_info = graph_data
    
    # Get predictions
    model.eval()
    with torch.no_grad():
        predictions = model(data.x, data.edge_index, data.edge_attr)
    
    # Map predictions back to (node, platform) combinations
    latency_predictions = {}
    for (node_name, platform_id), node_idx in node_mapping.items():
        latency_predictions[(node_name, platform_id)] = predictions[node_idx].item()
    
    return latency_predictions

def demonstrate_scheduler_usage(model, task_df, platform_df, node_df, system_df):
    """
    Demonstrate how the trained model would be used in the scheduler
    """
    print("\n" + "="*60)
    print("SCHEDULER USAGE DEMONSTRATION")
    print("="*60)
    
    # Get a sample task and system state
    sample_task = task_df.iloc[0]
    sample_system_state = system_df.iloc[0]['state']
    
    print(f"Sample Task: {sample_task['task_id']} ({sample_task['task_type']})")
    print(f"Source Node: {sample_task['source_node']}")
    print(f"Actual Execution Node: {sample_task['execution_node']}")
    print(f"Actual Latency: {sample_task['end_to_end_latency']:.2f} ms")
    
    # Get latency predictions for all available nodes
    latency_predictions = predict_latency_for_scheduling(
        model, sample_system_state, sample_task, task_df, platform_df, node_df
    )
    
    if latency_predictions:
        print("\nPredicted Latencies for Available (Node, Platform) Combinations:")
        print("-" * 80)
        
        # Sort by predicted latency
        sorted_predictions = sorted(latency_predictions.items(), key=lambda x: x[1])
        
        for i, ((node_name, platform_id), predicted_latency) in enumerate(sorted_predictions[:10]):
            # Get platform type
            platform_info = platform_df[
                (platform_df['node_id'] == node_name) & 
                (platform_df['platform_id'] == platform_id)
            ]
            platform_type = platform_info.iloc[0]['platform_type'] if not platform_info.empty else 'Unknown'
            
            # Check if this was the actual execution choice
            is_actual = (node_name == sample_task['execution_node'])
            actual_marker = " [ACTUAL]" if is_actual else ""
            
            print(f"{i+1:2d}. {node_name:6s} (Platform {platform_id:2d}, {platform_type:10s}): {predicted_latency:8.2f} ms{actual_marker}")
        
        # Find the best predicted choice
        best_node, best_platform = sorted_predictions[0][0]
        best_latency = sorted_predictions[0][1]
        
        print(f"\nBest Predicted Choice: {best_node} (Platform {best_platform}) - {best_latency:.2f} ms")
        print(f"Actual Choice: {sample_task['execution_node']} - {sample_task['end_to_end_latency']:.2f} ms")
        
        if best_node == sample_task['execution_node']:
            print("✅ Model correctly identified the best choice!")
        else:
            print(f"❌ Model chose {best_node} instead of {sample_task['execution_node']}")
            
        # Calculate prediction error
        actual_latency = sample_task['end_to_end_latency']
        prediction_error = abs(best_latency - actual_latency) / actual_latency * 100
        print(f"Prediction Error: {prediction_error:.1f}%")
    
    else:
        print("No predictions could be made. Check the system state structure.")

# Run the demonstration if we have a trained model
if 'model' in locals() and training_data:
    demonstrate_scheduler_usage(model, task_df, platform_df, node_df, system_df)
else:
    print("\nSkipping scheduler demonstration - no trained model available.")

# Data Analysis and Feature Engineering
print("\n" + "="*60)
print("DATA ANALYSIS AND FEATURE ENGINEERING")
print("="*60)

print(f"\nDataset Overview:")
print(f"- Tasks: {len(task_df)} total tasks")
print(f"- Nodes: {len(node_df)} total nodes")
print(f"- Platforms: {len(platform_df)} total platforms")
print(f"- System States: {len(system_df)} total snapshots")

print(f"\nTask Types:")
print(task_df['task_type'].value_counts())

print(f"\nPlatform Types:")
print(platform_df['platform_type'].value_counts())

print(f"\nExecution Nodes:")
print(task_df['execution_node'].value_counts())

print(f"\nLatency Statistics:")
print(task_df['end_to_end_latency'].describe())

# Analyze feature correlations
print(f"\n" + "-"*40)
print("FEATURE ANALYSIS")
print("-"*40)

# Show available features for nodes
print(f"\nNode Features Available:")
node_features = [col for col in node_df.columns if col not in ['experiment', 'file', 'node_id']]
for feature in node_features:
    print(f"  - {feature}")

# Show available features for platforms
print(f"\nPlatform Features Available:")
platform_features = [col for col in platform_df.columns if col not in ['experiment', 'file', 'node_id', 'platform_id']]
for feature in platform_features:
    print(f"  - {feature}")

# Show available features for tasks
print(f"\nTask Features Available:")
task_features = [col for col in task_df.columns if col not in ['experiment', 'file', 'task_id']]
for feature in task_features:
    print(f"  - {feature}")

# Analyze latency distribution by platform type
print(f"\n" + "-"*40)
print("LATENCY BY PLATFORM TYPE")
print("-"*40)

latency_by_platform = task_df.groupby('platform_type')['end_to_end_latency'].agg(['mean', 'std', 'count'])
print(latency_by_platform)

# Analyze latency distribution by execution node
print(f"\n" + "-"*40)
print("LATENCY BY EXECUTION NODE")
print("-"*40)

latency_by_node = task_df.groupby('execution_node')['end_to_end_latency'].agg(['mean', 'std', 'count'])
print(latency_by_node)

# Visualize latency distributions
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
task_df['end_to_end_latency'].hist(bins=50, alpha=0.7)
plt.xlabel('End-to-End Latency (ms)')
plt.ylabel('Frequency')
plt.title('Overall Latency Distribution')

plt.subplot(1, 3, 2)
task_df.boxplot(column='end_to_end_latency', by='platform_type', ax=plt.gca())
plt.xlabel('Platform Type')
plt.ylabel('End-to-End Latency (ms)')
plt.title('Latency by Platform Type')
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
task_df.boxplot(column='end_to_end_latency', by='execution_node', ax=plt.gca())
plt.xlabel('Execution Node')
plt.ylabel('End-to-End Latency (ms)')
plt.title('Latency by End-to-End Latency')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print("\nFeature Engineering Summary:")
print("1. Node Features: Platform price, energy metrics, scheduling time, storage time, dependencies, cache hits")
print("2. Platform Features: Hardware type (one-hot encoded), energy, idle time, idle proportion")
print("3. Task Features: Task type (one-hot encoded), dispatched time (normalized)")
print("4. Edge Features: Network latency between nodes")
print("5. Target: End-to-end latency for executed (node, platform) combination")
print("\nTraining Strategy:")
print("- Use execution mode data as ground truth (masked training)")
print("- Train on all nodes to learn global function")
print("- Predict latency for all available (node, platform) combinations")
print("- Use predictions to rank and select best placement option")